In [1]:
import pandas as pd
import numpy as np
import re
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
import tensorflow as tf


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


## Load Dataset

In [2]:
data = pd.read_csv('/content/Fake.csv')
print("Shape:", data.shape)
print("\nSubject distribution:")
print(data['subject'].value_counts())


Shape: (23481, 4)

Subject distribution:
subject
News               9050
politics           6841
left-news          4459
Government News    1570
US_News             783
Middle-east         778
Name: count, dtype: int64


In [3]:
data.head()


,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


## Label Encoding
The dataset has 6 subject categories. We encode them as integer labels for multiclass classification.

| Subject | Label |
|---|---|
| News | 0 |
| politics | 1 |
| left-news | 2 |
| Government News | 3 |
| US_News | 4 |
| Middle-east | 5 |


In [4]:
le = LabelEncoder()
data['label'] = le.fit_transform(data['subject'])

print("Label classes:", list(le.classes_))
print("Label mapping:")
for cls, idx in zip(le.classes_, le.transform(le.classes_)):
    print(f"  {cls:20s} → {idx}")

NUM_CLASSES = len(le.classes_)
print(f"\nTotal classes: {NUM_CLASSES}")


Label classes: ['Government News', 'Middle-east', 'News', 'US_News', 'left-news', 'politics']
Label mapping:
  Government News      → 0
  Middle-east          → 1
  News                 → 2
  US_News              → 3
  left-news            → 4
  politics             → 5

Total classes: 6


##  Text Preprocessing

In [5]:
stops = set(stopwords.words('english'))
ps    = PorterStemmer()

corpus = []
labels_list = []

for i in range(len(data)):
    text = re.sub('[^a-zA-Z]', ' ', str(data['text'][i]))
    text = text.lower().split()
    text = [ps.stem(w) for w in text if w not in stops]
    corpus.append(' '.join(text))
    labels_list.append(data['label'][i])

print(f"Corpus size: {len(corpus)}")
print(f"Sample: {corpus[0][:100]}")


Corpus size: 23481
Sample: donald trump wish american happi new year leav instead give shout enemi hater dishonest fake news me


## Hyperparameters

In [6]:
VOCAB_SIZE  = 10000
MAX_LEN     = 300
EMBED_DIM   = 128
BATCH_SIZE  = 64
EPOCHS      = 15


## Tokenization & Padding

In [7]:
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(corpus)

sequences = tokenizer.texts_to_sequences(corpus)
padded    = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

labels_arr = np.array(labels_list)
labels_cat  = to_categorical(labels_arr, num_classes=NUM_CLASSES)

print("Padded shape :", padded.shape)
print("Labels shape :", labels_cat.shape)


Padded shape : (23481, 300)
Labels shape : (23481, 6)


## Train / Test Split

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    padded, labels_cat,
    test_size=0.2, random_state=42, stratify=labels_arr
)

# Keep integer labels for evaluation metrics
_, _, y_train_int, y_test_int = train_test_split(
    padded, labels_arr,
    test_size=0.2, random_state=42, stratify=labels_arr
)

print(f"Train size : {X_train.shape}")
print(f"Test  size : {X_test.shape}")


Train size : (18784, 300)
Test  size : (4697, 300)


## Model 1 — LSTM

In [9]:
lstm_model = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM),
    LSTM(128, return_sequences=True),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(NUM_CLASSES, activation='softmax')      # multiclass output
])

lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
lstm_model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 300, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 300, 128)       │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 300, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,465,542 (5.59 MB)

 Trainable params: 1,465,542 (5.59 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history_LSTM = lstm_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop]
)


Epoch 1/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 310s 1s/step - accuracy: 0.3796 - loss: 1.4603 - val_accuracy: 0.3859 - val_loss: 1.4441
Epoch 2/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 253s 1s/step - accuracy: 0.3972 - loss: 1.4125 - val_accuracy: 0.3931 - val_loss: 1.3966
Epoch 3/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 260s 1s/step - accuracy: 0.5744 - loss: 1.0589 - val_accuracy: 0.6449 - val_loss: 0.8774
Epoch 4/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 252s 1s/step - accuracy: 0.6713 - loss: 0.7491 - val_accuracy: 0.6660 - val_loss: 0.7681
Epoch 5/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 265s 1s/step - accuracy: 0.6904 - loss: 0.6642 - val_accuracy: 0.6673 - val_loss: 0.7930
Epoch 6/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 253s 1s/step - accuracy: 0.6996 - loss: 0.6353 - val_accuracy: 0.6705 - val_loss: 0.7750
Epoch 7/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 257s 1s/step - accuracy: 0.7021 - loss: 0.6155 - val_accuracy: 0.6747 - val_loss: 0.7808


### LSTM — Evaluation

In [12]:
y_prob_lstm = lstm_model.predict(X_test)
y_pred_lstm = np.argmax(y_prob_lstm, axis=1)

print('\n' + '='*55)
print('  LSTM Results')
print('='*55)
print(classification_report(y_test_int, y_pred_lstm, target_names=le.classes_))
print('Confusion Matrix:')
print(confusion_matrix(y_test_int, y_pred_lstm))
print(f'\nOverall Accuracy: {accuracy_score(y_test_int, y_pred_lstm):.4f}')


147/147 ━━━━━━━━━━━━━━━━━━━━ 27s 185ms/step

  LSTM Results
                 precision    recall  f1-score   support

Government News       0.00      0.00      0.00       314
    Middle-east       0.48      0.79      0.60       156
           News       0.94      0.95      0.95      1810
        US_News       0.17      0.04      0.07       157
      left-news       0.24      0.01      0.02       892
       politics       0.50      0.93      0.65      1368

       accuracy                           0.67      4697
      macro avg       0.39      0.45      0.38      4697
   weighted avg       0.57      0.67      0.58      4697

Confusion Matrix:
[[   0    0    7    3    6  298]
 [   0  124    3    8    2   19]
 [   0    3 1721    1    7   78]
 [   0  122    5    7    1   22]
 [   0    3   32    7   10  840]
 [   0    7   64   15   16 1266]]

Overall Accuracy: 0.6660


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


##Model 2 — Bidirectional LSTM

In [13]:
BI_lstm_model = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM),
    Bidirectional(LSTM(128, return_sequences=True)),
    Dropout(0.3),
    Bidirectional(LSTM(64)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(NUM_CLASSES, activation='softmax')      # multiclass output
])

BI_lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
BI_lstm_model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 300, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 300, 256)       │       263,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 300, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,716,166 (6.55 MB)

 Trainable params: 1,716,166 (6.55 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
history_BI = BI_lstm_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop]
)


Epoch 1/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 569s 2s/step - accuracy: 0.5828 - loss: 1.0051 - val_accuracy: 0.6550 - val_loss: 0.7924
Epoch 2/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 611s 2s/step - accuracy: 0.6754 - loss: 0.7063 - val_accuracy: 0.6590 - val_loss: 0.7947
Epoch 3/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 563s 2s/step - accuracy: 0.6936 - loss: 0.6214 - val_accuracy: 0.6614 - val_loss: 0.7821


### Bi-LSTM — Evaluation

In [15]:
y_prob_bi = BI_lstm_model.predict(X_test)
y_pred_bi = np.argmax(y_prob_bi, axis=1)

print('\n' + '='*55)
print('  Bi-LSTM Results')
print('='*55)
print(classification_report(y_test_int, y_pred_bi, target_names=le.classes_))
print('Confusion Matrix:')
print(confusion_matrix(y_test_int, y_pred_bi))
print(f'\nOverall Accuracy: {accuracy_score(y_test_int, y_pred_bi):.4f}')


147/147 ━━━━━━━━━━━━━━━━━━━━ 73s 483ms/step

  Bi-LSTM Results
                 precision    recall  f1-score   support

Government News       0.00      0.00      0.00       314
    Middle-east       0.45      0.67      0.54       156
           News       0.86      0.94      0.90      1810
        US_News       0.42      0.20      0.27       157
      left-news       0.00      0.00      0.00       892
       politics       0.50      0.88      0.64      1368

       accuracy                           0.65      4697
      macro avg       0.37      0.45      0.39      4697
   weighted avg       0.50      0.65      0.56      4697

Confusion Matrix:
[[   0    0   32    1    0  281]
 [   0  105    6   35    0   10]
 [   0    0 1694    1    0  115]
 [   0  118    4   31    0    4]
 [   0    4   84    3    0  801]
 [   0    4  155    3    0 1206]]

Overall Accuracy: 0.6464


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Custom News Prediction — Both Models

In [16]:
def preprocess_text(text):
    text = re.sub('[^a-zA-Z]', ' ', str(text))
    text = text.lower().split()
    text = [ps.stem(w) for w in text if w not in stops]
    return ' '.join(text)

def predict_news(text):
    """Predict news category using both LSTM and Bi-LSTM."""
    cleaned = preprocess_text(text)
    seq     = tokenizer.texts_to_sequences([cleaned])
    padded  = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')

    # ── LSTM ──────────────────────────────────────────────────────
    lstm_probs  = lstm_model.predict(padded, verbose=0)[0]
    lstm_idx    = np.argmax(lstm_probs)
    lstm_label  = le.classes_[lstm_idx]
    lstm_conf   = lstm_probs[lstm_idx]

    # ── Bi-LSTM ───────────────────────────────────────────────────
    bi_probs    = BI_lstm_model.predict(padded, verbose=0)[0]
    bi_idx      = np.argmax(bi_probs)
    bi_label    = le.classes_[bi_idx]
    bi_conf     = bi_probs[bi_idx]

    print(f'\n📰 News: "{text[:80]}..."' if len(text) > 80 else f'\n📰 News: "{text}"')
    print('─' * 60)
    print(f'  LSTM     → {lstm_label:20s}  (confidence: {lstm_conf:.4f})')
    print(f'  Bi-LSTM  → {bi_label:20s}  (confidence: {bi_conf:.4f})')
    print('─' * 60)


### Sample Predictions by Category

In [17]:
# News
predict_news("The stock market dropped significantly today amid global economic concerns and inflation fears.")

# Politics
predict_news("The senator proposed a new bill to reform healthcare policies across the country.")

# Government News
predict_news("The federal government announced a new budget plan with increased spending on infrastructure.")

# Left-News
predict_news("Progressive activists rallied for climate change legislation and social justice reforms.")

# US News
predict_news("A major hurricane made landfall in Florida causing widespread damage to coastal communities.")

# Middle-east
predict_news("Tensions escalated in the region as military forces moved toward the border.")



📰 News: "The stock market dropped significantly today amid global economic concerns and i..."
────────────────────────────────────────────────────────────
  LSTM     → politics              (confidence: 0.5199)
  Bi-LSTM  → politics              (confidence: 0.6602)
────────────────────────────────────────────────────────────

📰 News: "The senator proposed a new bill to reform healthcare policies across the country..."
────────────────────────────────────────────────────────────
  LSTM     → politics              (confidence: 0.5596)
  Bi-LSTM  → politics              (confidence: 0.6577)
────────────────────────────────────────────────────────────

📰 News: "The federal government announced a new budget plan with increased spending on in..."
────────────────────────────────────────────────────────────
  LSTM     → politics              (confidence: 0.5563)
  Bi-LSTM  → politics              (confidence: 0.6603)
────────────────────────────────────────────────────────────

📰 News: "Pro

### Batch Prediction Comparison Table

In [18]:
test_news = [
    "The White House released a statement on foreign trade agreements today.",
    "Lawmakers debated the immigration reform bill in the Senate chambers.",
    "Government officials confirmed new regulations on environmental standards.",
    "Left-wing groups organized protests outside city hall demanding change.",
    "A shooting incident in Texas left several people injured, police confirm.",
    "Ceasefire talks between rival factions in the Middle East collapsed.",
    "The president signed an executive order on national security measures.",
    "Economic sanctions were imposed on the country following human rights concerns.",
]

rows = []
for news in test_news:
    cleaned = preprocess_text(news)
    seq     = tokenizer.texts_to_sequences([cleaned])
    pad     = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')

    lp  = lstm_model.predict(pad, verbose=0)[0]
    bp  = BI_lstm_model.predict(pad, verbose=0)[0]

    l_label = le.classes_[np.argmax(lp)]
    b_label = le.classes_[np.argmax(bp)]

    rows.append({
        'News (truncated)'  : news[:50] + '...',
        'LSTM Category'     : l_label,
        'LSTM Conf'         : round(float(np.max(lp)), 4),
        'Bi-LSTM Category'  : b_label,
        'Bi-LSTM Conf'      : round(float(np.max(bp)), 4),
        'Agree?'            : '✅' if l_label == b_label else '⚠️',
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))


                                     News (truncated) LSTM Category  LSTM Conf Bi-LSTM Category  Bi-LSTM Conf Agree?
The White House released a statement on foreign tr...      politics     0.5342         politics        0.6656      ✅
Lawmakers debated the immigration reform bill in t...      politics     0.5596         politics        0.6693      ✅
Government officials confirmed new regulations on ...      politics     0.5606         politics        0.6664      ✅
Left-wing groups organized protests outside city h...      politics     0.5597         politics        0.6630      ✅
A shooting incident in Texas left several people i...      politics     0.5564         politics        0.6568      ✅
Ceasefire talks between rival factions in the Midd...      politics     0.5583         politics        0.6656      ✅
The president signed an executive order on nationa...       US_News     0.3949         politics        0.6672     ⚠️
Economic sanctions were imposed on the country fol...       US_N

### Try Own News Text

In [19]:
my_news = "The government passed a new policy on education funding for public schools."

predict_news(my_news)



📰 News: "The government passed a new policy on education funding for public schools."
────────────────────────────────────────────────────────────
  LSTM     → politics              (confidence: 0.5581)
  Bi-LSTM  → politics              (confidence: 0.6613)
────────────────────────────────────────────────────────────
